In [1]:
!pip install feedparser newspaper3k pandas requests beautifulsoup4 lxml_html_clean Sastrawi
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt to C:\Users\HP
[nltk_data]     840\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\HP
[nltk_data]     840\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\HP
[nltk_data]     840\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

**Upload file sraped_articles_merged.csv terlebih dahulu**

### Handling Missing Values, Lowercasing, Special Char Removal, Punctuations Removal

In [12]:
import pandas as pd
import re
import numpy as np

file_merged = '../Data/scraped_articles_merged.csv'
try:
    df_result = pd.read_csv(file_merged)
    print(f"File '{file_merged}' berhasil dimuat. Total: {len(df_result)} baris.")
except FileNotFoundError:
    print(f"Error: File tidak ditemukan di {file_merged}. Pastikan file sudah tersedia.")
    df_result = pd.DataFrame()

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Normalisasi istilah
    text = text.lower()
    text = text.replace('e-commerce', 'ecommerce')
    text = text.replace('tiktok shop', 'tiktokshop')

    # Menghapus teks dalam kurung
    text = re.sub(r'\([^)]*\)', '', text)
    # Menghapus url
    text = re.sub(r'http[s]?://\S+', '', text)
    # Menghapus kata penerbit, kota di awal (sebelum tanda pisah)
    text = re.sub(r'^[^—-]+[-—]\s*', '', text)
    # Menghapus tanggal di awal
    text = re.sub(r'^\d{1,2}\s\w+\s\d{4}\s*[-:]\s*', '', text)
    # Karakter html
    text = re.sub(r'&\w+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)
    # Menghapus angka
    text = re.sub(r'\d+', '', text)
    # Hilangkan emoji dan simbol unicode lainnya
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # Hilangkan tanda baca dan simbol matematika
    text = re.sub(r'[^\w\s]', ' ', text)
    return text

if not df_result.empty:
    df_result['content'] = df_result['content'].fillna('')

    df_result['content_cleaned'] = df_result['content'].apply(clean_text)

    df_result['content_cleaned'] = df_result['content_cleaned'].str.lower()

    df_result['content_cleaned'] = df_result['content_cleaned'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

    initial_rows = len(df_result)
    df_result = df_result[df_result['content_cleaned'].str.strip() != '']
    rows_removed = initial_rows - len(df_result)

    print(f"Removed {rows_removed} rows with empty content_cleaned after cleaning.")
    display(df_result[['title', 'content', 'content_cleaned']].head())
else:
    print("Tidak ada data untuk diproses.")

File '../Data/scraped_articles_merged.csv' berhasil dimuat. Total: 1053 baris.
Removed 249 rows with empty content_cleaned after cleaning.


,title,content,content_cleaned
0,TikTok raises minimum user age in Indonesia fo...,Jakarta (ANTARA) -\nSocial media giant TikTok ...,social media giant tiktok announced that it ha...
1,TikTok Indonesia Resmi Umumkan Tutup Fitur Tik...,Fitur TikTok Shop resmi ditutup hari ini./\nJa...,tiktokshop mulai hari ini rabu tepatnya pada p...
3,LIVE Tokopedia–TikTok Shop Ditonton 38 Miliar ...,Kementerian Perdagangan mencatat nilai penjual...,kementerian perdagangan mencatat nilai penjual...
4,Konten Jadi Mesin Penggerak Penjualan di Tokop...,Reporter: Dina Mirayanti Hutauruk | Editor: Di...,jakarta tren ecommerce saat ini mengalami perg...
5,usmile Indonesia Berhasil Dominasi Kategori Pr...,@2026 marketing.co.id – All Right Reserved.\nM...,rata pesanan harian hingga dibandingkan period...


### Tokenization

In [13]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

def tokenize_text(text):
    if not isinstance(text, str):
        return []
    return word_tokenize(text)

if 'content_cleaned' in df_result.columns:
    df_result['tokenization'] = df_result['content_cleaned'].apply(tokenize_text)
    display(df_result[['title', 'content_cleaned', 'tokenization']].head())
else:
    print("Error: Kolom 'content_cleaned' tidak ditemukan. Jalankan preprocessing terlebih dahulu.")

[nltk_data] Downloading package punkt to C:\Users\HP
[nltk_data]     840\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\HP
[nltk_data]     840\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,title,content_cleaned,tokenization
0,TikTok raises minimum user age in Indonesia fo...,social media giant tiktok announced that it ha...,"[social, media, giant, tiktok, announced, that..."
1,TikTok Indonesia Resmi Umumkan Tutup Fitur Tik...,tiktokshop mulai hari ini rabu tepatnya pada p...,"[tiktokshop, mulai, hari, ini, rabu, tepatnya,..."
3,LIVE Tokopedia–TikTok Shop Ditonton 38 Miliar ...,kementerian perdagangan mencatat nilai penjual...,"[kementerian, perdagangan, mencatat, nilai, pe..."
4,Konten Jadi Mesin Penggerak Penjualan di Tokop...,jakarta tren ecommerce saat ini mengalami perg...,"[jakarta, tren, ecommerce, saat, ini, mengalam..."
5,usmile Indonesia Berhasil Dominasi Kategori Pr...,rata pesanan harian hingga dibandingkan period...,"[rata, pesanan, harian, hingga, dibandingkan, ..."


### Stopwords Removal

In [14]:
import nltk
from nltk.corpus import stopwords

try:
    stopwords.words('indonesian')
except LookupError:
    nltk.download('stopwords')

stop_words_indonesian = set(stopwords.words('indonesian'))
custom_stopwords = {'nya', 'akan', 'juga', 'pun', 'saja', 'tersebut', 'ini', 'itu', 'adalah', 'dengan', 'dari', 'yang', 'editor', 'reporter'}
stop_words_indonesian.update(custom_stopwords)

def remove_stopwords_from_tokens(tokens_list):
    return [word for word in tokens_list if word not in stop_words_indonesian]

if 'tokenization' in df_result.columns:
    df_result['stopword_removed'] = df_result['tokenization'].apply(remove_stopwords_from_tokens)
    print("Stopword removal selesai. Kolom 'stopword_removed' telah ditambahkan.")
    display(df_result[['title', 'tokenization', 'stopword_removed']].head())
else:
    print("Error: Kolom 'tokenization' tidak ditemukan.")

Stopword removal selesai. Kolom 'stopword_removed' telah ditambahkan.


,title,tokenization,stopword_removed
0,TikTok raises minimum user age in Indonesia fo...,"[social, media, giant, tiktok, announced, that...","[social, media, giant, tiktok, announced, that..."
1,TikTok Indonesia Resmi Umumkan Tutup Fitur Tik...,"[tiktokshop, mulai, hari, ini, rabu, tepatnya,...","[tiktokshop, rabu, tepatnya, wib, resmi, tutup..."
3,LIVE Tokopedia–TikTok Shop Ditonton 38 Miliar ...,"[kementerian, perdagangan, mencatat, nilai, pe...","[kementerian, perdagangan, mencatat, nilai, pe..."
4,Konten Jadi Mesin Penggerak Penjualan di Tokop...,"[jakarta, tren, ecommerce, saat, ini, mengalam...","[jakarta, tren, ecommerce, mengalami, pergeser..."
5,usmile Indonesia Berhasil Dominasi Kategori Pr...,"[rata, pesanan, harian, hingga, dibandingkan, ...","[pesanan, harian, dibandingkan, periode, ramad..."


### Stemming

In [15]:
!pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens_list):
    text = " ".join(tokens_list)
    stemmed_text = stemmer.stem(text)
    # Mengembalikan dalam bentuk list token lagi
    return stemmed_text.split()

if 'stopword_removed' in df_result.columns:
    df_result['stemming_output'] = df_result['stopword_removed'].apply(stem_tokens)
    print("Stemming selesai. Kolom 'stemming_output' telah ditambahkan.")
    display(df_result[['title', 'stopword_removed', 'stemming_output']].head())
else:
    print("Error: Kolom 'stopword_removed' tidak ditemukan.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Stemming selesai. Kolom 'stemming_output' telah ditambahkan.


,title,stopword_removed,stemming_output
0,TikTok raises minimum user age in Indonesia fo...,"[social, media, giant, tiktok, announced, that...","[social, media, giant, tiktok, announced, that..."
1,TikTok Indonesia Resmi Umumkan Tutup Fitur Tik...,"[tiktokshop, rabu, tepatnya, wib, resmi, tutup...","[tiktokshop, rabu, tepat, wib, resmi, tutup, u..."
3,LIVE Tokopedia–TikTok Shop Ditonton 38 Miliar ...,"[kementerian, perdagangan, mencatat, nilai, pe...","[menteri, dagang, catat, nilai, jual, ecommerc..."
4,Konten Jadi Mesin Penggerak Penjualan di Tokop...,"[jakarta, tren, ecommerce, mengalami, pergeser...","[jakarta, tren, ecommerce, alami, geser, konsu..."
5,usmile Indonesia Berhasil Dominasi Kategori Pr...,"[pesanan, harian, dibandingkan, periode, ramad...","[pesan, hari, banding, periode, ramadan, tren,..."


### Rare Words Removal dari Stemming

In [16]:
from collections import Counter

all_tokens = [token for sublist in df_result['stemming_output'] for token in sublist]
word_freq = Counter(all_tokens)

rare_threshold = 3 # ambang batas
rare_words = {word for word, count in word_freq.items() if count <= rare_threshold}

def remove_rare_words(tokens_list):
    return [word for word in tokens_list if word not in rare_words]

if 'stemming_output' in df_result.columns:
    df_result['stemming_cleaned'] = df_result['stemming_output'].apply(remove_rare_words)

    print(f"Rare words removal selesai.")
    print(f"Jumlah rare words ditemukan (freq <= {rare_threshold}): {len(rare_words)}")
    print("Kolom 'stemming_cleaned' telah ditambahkan sebagai hasil akhir preprocessing.")
    display(df_result[['title', 'stemming_output', 'stemming_cleaned']].head())
else:
    print("Error: Kolom 'stemming_output' tidak ditemukan.")

Rare words removal selesai.
Jumlah rare words ditemukan (freq <= 3): 6471
Kolom 'stemming_cleaned' telah ditambahkan sebagai hasil akhir preprocessing.


,title,stemming_output,stemming_cleaned
0,TikTok raises minimum user age in Indonesia fo...,"[social, media, giant, tiktok, announced, that...","[social, media, giant, tiktok, that, it, has, ..."
1,TikTok Indonesia Resmi Umumkan Tutup Fitur Tik...,"[tiktokshop, rabu, tepat, wib, resmi, tutup, u...","[tiktokshop, rabu, tepat, wib, resmi, tutup, u..."
3,LIVE Tokopedia–TikTok Shop Ditonton 38 Miliar ...,"[menteri, dagang, catat, nilai, jual, ecommerc...","[menteri, dagang, catat, nilai, jual, ecommerc..."
4,Konten Jadi Mesin Penggerak Penjualan di Tokop...,"[jakarta, tren, ecommerce, alami, geser, konsu...","[jakarta, tren, ecommerce, alami, geser, konsu..."
5,usmile Indonesia Berhasil Dominasi Kategori Pr...,"[pesan, hari, banding, periode, ramadan, tren,...","[pesan, hari, banding, periode, ramadan, tren,..."


### OUTPUT

In [17]:
output_file_path = 'scraped_articles_preprocessed.csv'
df_result.to_csv(output_file_path, index=False)
print(f"File preprocessed berhasil disimpan ke: {output_file_path}")

columns_to_show = ['title', 'content_cleaned', 'tokenization', 'stopword_removed', 'stemming_output', 'stemming_cleaned']
display(df_result[columns_to_show].head())

File preprocessed berhasil disimpan ke: scraped_articles_preprocessed.csv


,title,content_cleaned,tokenization,stopword_removed,stemming_output,stemming_cleaned
0,TikTok raises minimum user age in Indonesia fo...,social media giant tiktok announced that it ha...,"[social, media, giant, tiktok, announced, that...","[social, media, giant, tiktok, announced, that...","[social, media, giant, tiktok, announced, that...","[social, media, giant, tiktok, that, it, has, ..."
1,TikTok Indonesia Resmi Umumkan Tutup Fitur Tik...,tiktokshop mulai hari ini rabu tepatnya pada p...,"[tiktokshop, mulai, hari, ini, rabu, tepatnya,...","[tiktokshop, rabu, tepatnya, wib, resmi, tutup...","[tiktokshop, rabu, tepat, wib, resmi, tutup, u...","[tiktokshop, rabu, tepat, wib, resmi, tutup, u..."
3,LIVE Tokopedia–TikTok Shop Ditonton 38 Miliar ...,kementerian perdagangan mencatat nilai penjual...,"[kementerian, perdagangan, mencatat, nilai, pe...","[kementerian, perdagangan, mencatat, nilai, pe...","[menteri, dagang, catat, nilai, jual, ecommerc...","[menteri, dagang, catat, nilai, jual, ecommerc..."
4,Konten Jadi Mesin Penggerak Penjualan di Tokop...,jakarta tren ecommerce saat ini mengalami perg...,"[jakarta, tren, ecommerce, saat, ini, mengalam...","[jakarta, tren, ecommerce, mengalami, pergeser...","[jakarta, tren, ecommerce, alami, geser, konsu...","[jakarta, tren, ecommerce, alami, geser, konsu..."
5,usmile Indonesia Berhasil Dominasi Kategori Pr...,rata pesanan harian hingga dibandingkan period...,"[rata, pesanan, harian, hingga, dibandingkan, ...","[pesanan, harian, dibandingkan, periode, ramad...","[pesan, hari, banding, periode, ramadan, tren,...","[pesan, hari, banding, periode, ramadan, tren,..."
